# Strict Video Protocol Result Analysis

Post-processing notebook for `benchmark_strict_video_protocol.ipynb` outputs. Edit the output directory in the first code cell, then run top to bottom to produce visual model comparisons, timing/FPS summaries, optional CUDA memory summaries, per-frame curves, and an `analysis_visuals/` packet beside the benchmark results.

## 0. User Settings

In [ ]:
from pathlib import Path
import json
import math
import re
import warnings
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display, Markdown
except Exception:
    def display(x):
        print(x)
    def Markdown(x):
        return x

warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 140)
plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 160,
    'axes.grid': True,
    'grid.alpha': 0.22,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Edit this path first. It may point at either current or legacy strict-protocol outputs.
STRICT_OUTPUT_DIR = Path('tmp_strict_smoke_20260614_000038/strict_video_protocol_20260614_000038')

ANALYSIS_OUTPUT_DIR = STRICT_OUTPUT_DIR / 'analysis_visuals'
PRIMARY_SCORE = 'fg_volume_dice'
SECONDARY_SCORE = 'fg_volume_iou'
LOWER_IS_BETTER = ['wall_time_sec', 'sec_per_frame', 'delta_peak_cuda_mem_allocated_mb']

MAX_HEATMAP_ROWS = 40
MAX_BAR_ROWS = 40
MAX_PER_FRAME_CASES = 3
TOP_FAILURE_ROWS = 50


## 1. Artifact Discovery and Schema Validation

In [ ]:
ARTIFACT_CANDIDATES = OrderedDict([
    ('results', ['per_case_results.csv', 'strict_video_protocol_results.csv']),
    ('summary_overall', ['summary_model_prompt_dataset_macro.csv', 'strict_video_summary_by_model.csv']),
    ('summary_by_dataset', ['summary_by_dataset.csv']),
    ('summary_by_modality', ['summary_by_modality.csv']),
    ('summary_by_task', ['summary_by_task.csv']),
    ('per_frame', ['per_frame_metrics.csv', 'strict_video_per_frame_metrics.csv']),
    ('prompt_audit', ['prompt_audit.csv', 'strict_video_prompt_audit.csv']),
    ('api_probe', ['strict_video_api_probe.csv']),
    ('memory', ['memory_measurements.csv']),
    ('memory_summary', ['memory_summary.csv']),
    ('efficiency', ['efficiency_by_frame_count.csv']),
    ('threshold_long', ['threshold_metrics_long.csv']),
    ('status_summary', ['status_summary.csv', 'smoke_status_summary.csv']),
    ('failures', ['failures.csv', 'smoke_failures.csv']),
    ('best_overall', ['best_thresholds_overall_dataset_macro.csv']),
    ('best_dataset', ['best_thresholds_by_dataset.csv']),
    ('best_modality', ['best_thresholds_by_modality.csv']),
    ('best_task', ['best_thresholds_by_task.csv']),
])


def resolve_output_dir(path):
    path = Path(path).expanduser()
    candidates = [path]
    if not path.is_absolute():
        candidates.extend([
            Path.cwd() / path,
            Path.cwd().parent / path,
            Path.cwd() / 'notebooks' / path,
        ])
    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if candidate.is_dir():
            return candidate
    searched = '\n'.join(f' - {p.resolve()}' for p in candidates)
    raise FileNotFoundError(f'Could not find STRICT_OUTPUT_DIR. Searched:\n{searched}')


def read_first_existing(base_dir, filenames, required=False):
    for filename in filenames:
        path = base_dir / filename
        if path.is_file():
            return pd.read_csv(path), path
    if required:
        raise FileNotFoundError(f'Missing required artifact. Tried: {filenames}')
    return pd.DataFrame(), None


def load_artifacts(output_dir):
    artifacts = {}
    status_rows = []
    for key, filenames in ARTIFACT_CANDIDATES.items():
        df, path = read_first_existing(output_dir, filenames, required=(key == 'results'))
        artifacts[key] = df
        status_rows.append({
            'artifact': key,
            'found': path is not None,
            'file': path.name if path is not None else '',
            'rows': int(len(df)),
            'columns': int(len(df.columns)) if path is not None else 0,
        })
    status_json = {}
    status_path = output_dir / 'benchmark_status.json'
    if status_path.is_file():
        with open(status_path, 'r', encoding='utf-8') as f:
            status_json = json.load(f)
        status_rows.append({'artifact': 'benchmark_status_json', 'found': True, 'file': status_path.name, 'rows': 1, 'columns': len(status_json)})
    else:
        status_rows.append({'artifact': 'benchmark_status_json', 'found': False, 'file': '', 'rows': 0, 'columns': 0})
    return artifacts, pd.DataFrame(status_rows), status_json


STRICT_OUTPUT_DIR = resolve_output_dir(STRICT_OUTPUT_DIR)
ANALYSIS_OUTPUT_DIR = STRICT_OUTPUT_DIR / 'analysis_visuals'
ANALYSIS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

artifacts, artifact_status, benchmark_status = load_artifacts(STRICT_OUTPUT_DIR)

display(Markdown(f'**Output directory:** `{STRICT_OUTPUT_DIR}`'))
display(Markdown(f'**Analysis output directory:** `{ANALYSIS_OUTPUT_DIR}`'))
display(artifact_status)


## 2. Shared Analysis Helpers

In [ ]:
ID_COLS = ['model', 'prompt_mode', 'protocol']
CASE_COLS = ['dataset_name', 'case_id']
SCORE_METRICS = [
    'dice', 'iou', 'fg_volume_dice', 'fg_volume_iou', 'fg_mean_frame_dice',
    'full_sequence_dice', 'full_sequence_iou', 'hd95', 'false_positive_rate',
    'false_negative_rate', 'precision_ppv', 'recall_sensitivity', 'specificity',
    'volumetric_similarity', 'empty_frame_false_positive_rate', 'empty_frame_specificity',
]
EFFICIENCY_METRICS = [
    'wall_time_sec', 'init_time_sec', 'prompt_time_sec', 'propagation_time_sec',
    'reset_time_sec', 'sec_per_frame', 'frames_per_sec', 'n_frames', 'n_prompt_frames',
    'yielded_frame_count', 'untracked_frame_count', 'param_count',
]
MEMORY_METRICS = [
    'peak_cuda_mem_allocated_mb', 'peak_cuda_mem_reserved_mb',
    'delta_peak_cuda_mem_allocated_mb', 'delta_peak_cuda_mem_reserved_mb',
    'delta_peak_cuda_mem_allocated_mb_per_frame', 'delta_peak_cuda_mem_reserved_mb_per_frame',
    'baseline_cuda_allocated_mb', 'baseline_cuda_reserved_mb',
    'final_cuda_allocated_mb', 'final_cuda_reserved_mb',
]
COUNT_METRICS = [
    'foreground_frame_count', 'empty_frame_count', 'pred_fg_count', 'gt_fg_count',
    'pred_fg_frac', 'gt_fg_frac', 'tp', 'fp', 'fn', 'tn',
]
NUMERIC_CANDIDATES = list(dict.fromkeys(SCORE_METRICS + EFFICIENCY_METRICS + MEMORY_METRICS + COUNT_METRICS + [
    'dim', 'dataset_index', 'anchor_frame', 'number_of_prompted_frames', 'sequence_length',
    'frame_idx', 'frame_dice', 'frame_iou', 'frame_pred_fg_count', 'frame_gt_fg_count',
    'frame_tp', 'frame_fp', 'frame_fn', 'frame_tn', 'load_time_sec', 'missing_keys',
    'unexpected_keys',
]))


def safe_name(value):
    text = str(value).strip()
    text = re.sub(r'[^A-Za-z0-9_.-]+', '_', text)
    return text.strip('_') or 'unknown'


def ensure_columns(df, columns, default='unknown'):
    for col in columns:
        if col not in df.columns:
            df[col] = default
    return df


def coerce_numeric(df, columns=None):
    if df is None or df.empty:
        return df
    columns = NUMERIC_CANDIDATES if columns is None else columns
    for col in columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df


def normalize_results(df):
    if df is None or df.empty:
        return pd.DataFrame()
    out = df.copy()
    out = ensure_columns(out, ['status'], default='ok')
    out = ensure_columns(out, ['dataset_name', 'case_id', 'modality', 'task_id', 'task_label', 'model', 'prompt_mode', 'protocol'])
    for col in ['dataset_name', 'case_id', 'modality', 'task_id', 'task_label', 'model', 'prompt_mode', 'protocol', 'status']:
        out[col] = out[col].fillna('unknown').astype(str).replace({'': 'unknown', 'nan': 'unknown'})
    out = coerce_numeric(out)

    if 'n_frames' not in out.columns:
        out['n_frames'] = np.nan
    if {'foreground_frame_count', 'empty_frame_count'}.issubset(out.columns):
        derived_frames = out['foreground_frame_count'].fillna(0) + out['empty_frame_count'].fillna(0)
        out['n_frames'] = out['n_frames'].where(out['n_frames'].notna() & (out['n_frames'] > 0), derived_frames)

    if 'sec_per_frame' not in out.columns:
        out['sec_per_frame'] = np.nan
    if 'frames_per_sec' not in out.columns:
        out['frames_per_sec'] = np.nan
    if 'wall_time_sec' in out.columns:
        valid = out['wall_time_sec'].notna() & out['n_frames'].notna() & (out['wall_time_sec'] > 0) & (out['n_frames'] > 0)
        out.loc[valid & out['sec_per_frame'].isna(), 'sec_per_frame'] = out.loc[valid, 'wall_time_sec'] / out.loc[valid, 'n_frames']
        out.loc[valid & out['frames_per_sec'].isna(), 'frames_per_sec'] = out.loc[valid, 'n_frames'] / out.loc[valid, 'wall_time_sec']

    if 'delta_peak_cuda_mem_allocated_mb_per_frame' not in out.columns:
        out['delta_peak_cuda_mem_allocated_mb_per_frame'] = np.nan
    if 'delta_peak_cuda_mem_allocated_mb' in out.columns:
        valid_mem = out['delta_peak_cuda_mem_allocated_mb'].notna() & out['n_frames'].notna() & (out['n_frames'] > 0)
        out.loc[valid_mem & out['delta_peak_cuda_mem_allocated_mb_per_frame'].isna(), 'delta_peak_cuda_mem_allocated_mb_per_frame'] = (
            out.loc[valid_mem, 'delta_peak_cuda_mem_allocated_mb'] / out.loc[valid_mem, 'n_frames']
        )

    out['run_label'] = out['model'] + ' | ' + out['prompt_mode'] + ' | ' + out['protocol']
    out['case_label'] = out['dataset_name'] + ' / ' + out['case_id']
    return out


def normalize_per_frame(df):
    if df is None or df.empty:
        return pd.DataFrame()
    out = normalize_results(df)
    out = coerce_numeric(out, ['frame_idx', 'frame_dice', 'frame_iou', 'frame_pred_fg_count', 'frame_gt_fg_count', 'frame_tp', 'frame_fp', 'frame_fn', 'frame_tn', 'anchor_frame'])
    return out


def available_metrics(df, metrics):
    if df is None or df.empty:
        return []
    return [c for c in metrics if c in df.columns and df[c].notna().any()]


def case_count_column(df):
    if 'dataset_index' in df.columns and df['dataset_index'].notna().any():
        return 'dataset_index'
    if 'case_id' in df.columns:
        return 'case_id'
    return None


def numeric_mean_summary(df, group_cols, metric_cols):
    if df is None or df.empty:
        return pd.DataFrame()
    work = df.copy()
    work = ensure_columns(work, group_cols)
    cols = available_metrics(work, metric_cols)
    grouped = work.groupby(group_cols, dropna=False)
    out = grouped[cols].mean(numeric_only=True).reset_index() if cols else grouped.size().reset_index(name='rows')
    count_col = case_count_column(work)
    if count_col is not None:
        counts = grouped[count_col].nunique().reset_index(name='n_cases')
        out = counts.merge(out, on=group_cols, how='left')
    else:
        counts = grouped.size().reset_index(name='n_cases')
        out = counts.merge(out, on=group_cols, how='left')
    return out


def dataset_macro_summary(df, group_cols, metric_cols):
    if df is None or df.empty:
        return pd.DataFrame()
    work = df.copy()
    work = ensure_columns(work, group_cols + ['dataset_name'])
    cols = available_metrics(work, metric_cols)
    if not cols:
        return numeric_mean_summary(work, group_cols, metric_cols)
    ds_cols = list(dict.fromkeys(group_cols + ['dataset_name']))
    per_dataset = work.groupby(ds_cols, dropna=False)[cols].mean(numeric_only=True).reset_index()
    out = per_dataset.groupby(group_cols, dropna=False)[cols].mean(numeric_only=True).reset_index()
    n_datasets = per_dataset.groupby(group_cols, dropna=False)['dataset_name'].nunique().reset_index(name='n_datasets')
    count_col = case_count_column(work)
    if count_col is not None:
        n_cases = work.groupby(group_cols, dropna=False)[count_col].nunique().reset_index(name='n_cases')
    else:
        n_cases = work.groupby(group_cols, dropna=False).size().reset_index(name='n_cases')
    return n_datasets.merge(n_cases, on=group_cols, how='left').merge(out, on=group_cols, how='left')


def save_table(df, filename):
    path = ANALYSIS_OUTPUT_DIR / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    (df if df is not None else pd.DataFrame()).to_csv(path, index=False)
    return path


def display_table(title, df, columns=None, sort_by=None, ascending=False, n=50, save_name=None):
    display(Markdown(f'### {title}'))
    if df is None or df.empty:
        display(Markdown('_No rows._'))
        return pd.DataFrame()
    work = df.copy()
    if sort_by is not None and sort_by in work.columns:
        work = work.sort_values(sort_by, ascending=ascending, na_position='last')
    if columns is not None:
        columns = [c for c in columns if c in work.columns]
        work = work.loc[:, columns]
    if save_name:
        save_table(work, save_name)
    shown = work.head(int(n))
    numeric_cols = shown.select_dtypes(include='number').columns.tolist()
    fmt = {c: '{:.4f}' for c in numeric_cols}
    try:
        display(shown.style.format(fmt, na_rep=''))
    except Exception:
        display(shown)
    if len(work) > len(shown):
        display(Markdown(f'_Showing {len(shown):,} of {len(work):,} rows._'))
    return work


def save_fig(fig, filename):
    path = ANALYSIS_OUTPUT_DIR / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(path, bbox_inches='tight')
    display(fig)
    plt.close(fig)
    print('Saved:', path)
    return path


## 3. Normalize Results and Build Summary Tables

In [ ]:
results = normalize_results(artifacts['results'])
per_frame = normalize_per_frame(artifacts.get('per_frame', pd.DataFrame()))
prompt_audit = normalize_results(artifacts.get('prompt_audit', pd.DataFrame()))
api_probe = artifacts.get('api_probe', pd.DataFrame()).copy()
memory_measurements = normalize_results(artifacts.get('memory', pd.DataFrame()))

ok_results = results[results['status'].astype(str).str.lower().eq('ok')].copy()
failed_results = results[~results['status'].astype(str).str.lower().eq('ok')].copy()

analysis_metrics = list(dict.fromkeys(SCORE_METRICS + EFFICIENCY_METRICS + MEMORY_METRICS + COUNT_METRICS))
overall = dataset_macro_summary(ok_results, ID_COLS, analysis_metrics)
summary_by_dataset = numeric_mean_summary(ok_results, ['dataset_name'] + ID_COLS, analysis_metrics)
summary_by_modality = dataset_macro_summary(ok_results, ['modality'] + ID_COLS, analysis_metrics)
summary_by_task = dataset_macro_summary(ok_results, ['task_id', 'task_label'] + ID_COLS, analysis_metrics)
timing_summary = overall.loc[:, [c for c in ID_COLS + ['n_datasets', 'n_cases', 'n_frames', 'wall_time_sec', 'sec_per_frame', 'frames_per_sec'] if c in overall.columns]].copy()

memory_source = memory_measurements if not memory_measurements.empty else ok_results
memory_cols = available_metrics(memory_source, MEMORY_METRICS)
if memory_cols:
    memory_summary = numeric_mean_summary(memory_source, ID_COLS, memory_cols + ['wall_time_sec', 'sec_per_frame', 'frames_per_sec', 'n_frames'])
else:
    memory_summary = pd.DataFrame()

saved_tables = {
    'overall_summary.csv': save_table(overall, 'overall_summary.csv'),
    'summary_by_dataset.csv': save_table(summary_by_dataset, 'summary_by_dataset.csv'),
    'summary_by_modality.csv': save_table(summary_by_modality, 'summary_by_modality.csv'),
    'summary_by_task.csv': save_table(summary_by_task, 'summary_by_task.csv'),
    'timing_summary.csv': save_table(timing_summary, 'timing_summary.csv'),
}
if not memory_summary.empty:
    saved_tables['memory_summary.csv'] = save_table(memory_summary, 'memory_summary.csv')

manifest = pd.DataFrame([{'name': k, 'path': str(v)} for k, v in saved_tables.items()])
save_table(manifest, 'analysis_manifest.csv')

display(Markdown(f'Loaded `{len(results):,}` result rows with `{len(ok_results):,}` successful rows and `{len(failed_results):,}` failed/non-ok rows.'))
display(Markdown(f'Found `{results["model"].nunique():,}` models, `{results["dataset_name"].nunique():,}` datasets, `{results["prompt_mode"].nunique():,}` prompt modes, and `{results["protocol"].nunique():,}` protocols.'))
display(manifest)


## 4. Run and Status Overview

In [ ]:
display_table('Artifact status', artifact_status, save_name='artifact_status.csv')

if benchmark_status:
    interesting_keys = [
        'run_name', 'output_dir', 'smoke_run', 'runs_done', 'runs_total', 'percent_done',
        'models_done_or_current', 'models_total', 'current_dataset', 'last_update',
    ]
    status_preview = pd.DataFrame([{k: benchmark_status.get(k, '') for k in interesting_keys if k in benchmark_status}])
    display_table('Benchmark status JSON', status_preview)

status_counts = results.groupby(ID_COLS + ['status'], dropna=False).size().reset_index(name='rows')
display_table('Status by model, prompt, and protocol', status_counts, sort_by='rows', ascending=False, save_name='status_counts.csv')

dataset_counts = ok_results.groupby(['dataset_name', 'modality'], dropna=False)['case_id'].nunique().reset_index(name='ok_cases') if not ok_results.empty else pd.DataFrame()
display_table('Successful cases by dataset and modality', dataset_counts, sort_by='ok_cases', ascending=False, save_name='dataset_counts.csv')


## 5. Overall Model Ranking

In [ ]:
ranking_cols = [
    'model', 'prompt_mode', 'protocol', 'n_datasets', 'n_cases', PRIMARY_SCORE, SECONDARY_SCORE,
    'fg_mean_frame_dice', 'full_sequence_dice', 'wall_time_sec', 'sec_per_frame', 'frames_per_sec',
    'delta_peak_cuda_mem_allocated_mb', 'delta_peak_cuda_mem_allocated_mb_per_frame',
]
ranked = overall.copy()
if PRIMARY_SCORE in ranked.columns:
    ranked = ranked.sort_values(PRIMARY_SCORE, ascending=False, na_position='last')

display(Markdown('### Overall ranking'))
if ranked.empty:
    display(Markdown('_No successful result rows were available._'))
else:
    shown_cols = [c for c in ranking_cols if c in ranked.columns]
    shown = ranked.loc[:, shown_cols].copy()
    save_table(shown, 'overall_ranking.csv')
    best_value = shown[PRIMARY_SCORE].max() if PRIMARY_SCORE in shown.columns else np.nan
    numeric_cols = shown.select_dtypes(include='number').columns.tolist()
    fmt = {c: '{:.4f}' for c in numeric_cols}

    def highlight_best(row):
        if PRIMARY_SCORE in row.index and pd.notna(row[PRIMARY_SCORE]) and np.isclose(row[PRIMARY_SCORE], best_value, equal_nan=False):
            return ['background-color: #dcfce7'] * len(row)
        return [''] * len(row)

    try:
        display(shown.style.format(fmt, na_rep='').apply(highlight_best, axis=1))
    except Exception:
        display(shown)


In [ ]:
PROMPT_COLORS = {
    'box': '#2563eb',
    'mask': '#16a34a',
    'mixed': '#f97316',
    'unknown': '#6b7280',
}


def color_for_prompt(value):
    return PROMPT_COLORS.get(str(value).lower(), '#64748b')


def plot_barh(df, metric, title, filename, lower_is_better=False, max_rows=MAX_BAR_ROWS):
    if df is None or df.empty or metric not in df.columns:
        display(Markdown(f'_Skipping `{title}` because `{metric}` is unavailable._'))
        return None
    work = df.dropna(subset=[metric]).copy()
    if work.empty:
        display(Markdown(f'_Skipping `{title}` because `{metric}` has no finite values._'))
        return None
    if 'run_label' not in work.columns:
        work['run_label'] = work['model'].astype(str) + ' | ' + work['prompt_mode'].astype(str) + ' | ' + work['protocol'].astype(str)
    work = work.sort_values(metric, ascending=lower_is_better, na_position='last').head(int(max_rows))
    fig_h = max(4.0, 0.34 * len(work) + 1.7)
    fig, ax = plt.subplots(figsize=(11, fig_h))
    colors = [color_for_prompt(x) for x in work.get('prompt_mode', pd.Series(['unknown'] * len(work)))]
    ax.barh(work['run_label'], work[metric], color=colors)
    ax.invert_yaxis()
    ax.set_title(title)
    ax.set_xlabel(metric.replace('_', ' '))
    ax.set_ylabel('')
    return save_fig(fig, filename)


plot_barh(ranked, PRIMARY_SCORE, 'Overall dataset-macro foreground Dice', 'overall_fg_volume_dice.png')
if SECONDARY_SCORE in ranked.columns:
    plot_barh(ranked, SECONDARY_SCORE, 'Overall dataset-macro foreground IoU', 'overall_fg_volume_iou.png')
if 'frames_per_sec' in ranked.columns:
    plot_barh(ranked, 'frames_per_sec', 'Overall frames per second', 'overall_frames_per_sec.png')
if 'sec_per_frame' in ranked.columns:
    plot_barh(ranked, 'sec_per_frame', 'Overall seconds per frame', 'overall_sec_per_frame.png', lower_is_better=True)


## 6. Dataset, Modality, and Task Breakdowns

In [ ]:
def plot_heatmap(df, row_col, metric, title, filename, max_rows=MAX_HEATMAP_ROWS, max_cols=24):
    if df is None or df.empty or row_col not in df.columns or metric not in df.columns:
        display(Markdown(f'_Skipping `{title}` because required columns are unavailable._'))
        return None
    work = df.dropna(subset=[metric]).copy()
    if work.empty:
        display(Markdown(f'_Skipping `{title}` because `{metric}` has no finite values._'))
        return None
    if 'run_label' not in work.columns:
        work['run_label'] = work['model'].astype(str) + ' | ' + work['prompt_mode'].astype(str) + ' | ' + work['protocol'].astype(str)
    grouped = work.groupby([row_col, 'run_label'], dropna=False)[metric].mean().reset_index()
    pivot = grouped.pivot(index=row_col, columns='run_label', values=metric)
    pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index]
    pivot = pivot.iloc[: int(max_rows), : int(max_cols)]
    if pivot.empty:
        display(Markdown(f'_Skipping `{title}` because the pivot table is empty._'))
        return None
    fig_w = max(8.0, min(24.0, 1.15 * pivot.shape[1] + 3.0))
    fig_h = max(4.0, min(22.0, 0.38 * pivot.shape[0] + 2.4))
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    values = pivot.to_numpy(dtype=float)
    im = ax.imshow(values, aspect='auto', cmap='viridis')
    ax.set_title(title)
    ax.set_xticks(np.arange(pivot.shape[1]))
    ax.set_xticklabels(pivot.columns, rotation=45, ha='right')
    ax.set_yticks(np.arange(pivot.shape[0]))
    ax.set_yticklabels(pivot.index)
    ax.set_xlabel('model | prompt | protocol')
    ax.set_ylabel(row_col.replace('_', ' '))
    cbar = fig.colorbar(im, ax=ax, fraction=0.028, pad=0.02)
    cbar.set_label(metric.replace('_', ' '))
    if pivot.size <= 120:
        for i in range(pivot.shape[0]):
            for j in range(pivot.shape[1]):
                val = values[i, j]
                if np.isfinite(val):
                    ax.text(j, i, f'{val:.3f}', ha='center', va='center', color='white' if val < np.nanmean(values) else 'black', fontsize=8)
    return save_fig(fig, filename)


display_table('Dataset summary', summary_by_dataset, columns=['dataset_name', 'model', 'prompt_mode', 'protocol', 'n_cases', PRIMARY_SCORE, SECONDARY_SCORE, 'wall_time_sec', 'frames_per_sec'], sort_by=PRIMARY_SCORE, ascending=False, save_name='dataset_summary_display.csv')
plot_heatmap(summary_by_dataset, 'dataset_name', PRIMARY_SCORE, 'Foreground Dice by dataset', 'heatmap_dataset_fg_volume_dice.png')
if SECONDARY_SCORE in summary_by_dataset.columns:
    plot_heatmap(summary_by_dataset, 'dataset_name', SECONDARY_SCORE, 'Foreground IoU by dataset', 'heatmap_dataset_fg_volume_iou.png')

display_table('Modality summary', summary_by_modality, columns=['modality', 'model', 'prompt_mode', 'protocol', 'n_datasets', 'n_cases', PRIMARY_SCORE, SECONDARY_SCORE, 'wall_time_sec', 'frames_per_sec'], sort_by=PRIMARY_SCORE, ascending=False, save_name='modality_summary_display.csv')
plot_heatmap(summary_by_modality, 'modality', PRIMARY_SCORE, 'Foreground Dice by modality', 'heatmap_modality_fg_volume_dice.png')

task_row_col = 'task_label' if 'task_label' in summary_by_task.columns else 'task_id'
display_table('Task summary', summary_by_task, columns=['task_id', 'task_label', 'model', 'prompt_mode', 'protocol', 'n_datasets', 'n_cases', PRIMARY_SCORE, SECONDARY_SCORE, 'wall_time_sec', 'frames_per_sec'], sort_by=PRIMARY_SCORE, ascending=False, save_name='task_summary_display.csv')
plot_heatmap(summary_by_task, task_row_col, PRIMARY_SCORE, 'Foreground Dice by task', 'heatmap_task_fg_volume_dice.png')


## 7. Accuracy vs Efficiency Tradeoffs

In [ ]:
def short_model_label(value, max_len=26):
    text = str(value)
    return text if len(text) <= max_len else text[: max_len - 1] + '...'


def plot_tradeoff(df, x_metric, y_metric, title, filename):
    if df is None or df.empty or x_metric not in df.columns or y_metric not in df.columns:
        display(Markdown(f'_Skipping `{title}` because required metrics are unavailable._'))
        return None
    work = df.dropna(subset=[x_metric, y_metric]).copy()
    if work.empty:
        display(Markdown(f'_Skipping `{title}` because there are no finite points._'))
        return None
    fig, ax = plt.subplots(figsize=(9.5, 6.5))
    for prompt, group in work.groupby('prompt_mode', dropna=False):
        ax.scatter(group[x_metric], group[y_metric], s=95, alpha=0.86, label=str(prompt), color=color_for_prompt(prompt), edgecolor='white', linewidth=0.8)
    for _, row in work.iterrows():
        label = short_model_label(row.get('model', 'model')) + ' / ' + str(row.get('prompt_mode', ''))
        ax.annotate(label, (row[x_metric], row[y_metric]), textcoords='offset points', xytext=(5, 5), fontsize=8)
    ax.set_title(title)
    ax.set_xlabel(x_metric.replace('_', ' '))
    ax.set_ylabel(y_metric.replace('_', ' '))
    ax.legend(title='prompt', loc='best')
    return save_fig(fig, filename)


plot_tradeoff(ranked, 'frames_per_sec', PRIMARY_SCORE, 'Accuracy vs frames per second', 'tradeoff_dice_vs_fps.png')
plot_tradeoff(ranked, 'sec_per_frame', PRIMARY_SCORE, 'Accuracy vs seconds per frame', 'tradeoff_dice_vs_sec_per_frame.png')
if not memory_summary.empty and 'delta_peak_cuda_mem_allocated_mb' in ranked.columns:
    plot_tradeoff(ranked, 'delta_peak_cuda_mem_allocated_mb', PRIMARY_SCORE, 'Accuracy vs delta peak CUDA memory', 'tradeoff_dice_vs_cuda_delta_mb.png')
else:
    display(Markdown('_CUDA memory tradeoff plot not available: no memory artifact or CUDA memory columns were found._'))


## 8. Timing, FPS, and Memory Analysis

In [ ]:
display_table('Timing summary', timing_summary, sort_by='sec_per_frame', ascending=True, save_name='timing_summary_display.csv')
plot_barh(timing_summary.assign(run_label=timing_summary['model'] + ' | ' + timing_summary['prompt_mode'] + ' | ' + timing_summary['protocol']) if not timing_summary.empty else timing_summary, 'wall_time_sec', 'Mean wall time per case', 'timing_wall_time_sec.png', lower_is_better=True)
plot_barh(timing_summary.assign(run_label=timing_summary['model'] + ' | ' + timing_summary['prompt_mode'] + ' | ' + timing_summary['protocol']) if not timing_summary.empty else timing_summary, 'sec_per_frame', 'Mean seconds per frame', 'timing_sec_per_frame.png', lower_is_better=True)
plot_barh(timing_summary.assign(run_label=timing_summary['model'] + ' | ' + timing_summary['prompt_mode'] + ' | ' + timing_summary['protocol']) if not timing_summary.empty else timing_summary, 'frames_per_sec', 'Mean frames per second', 'timing_frames_per_sec.png')

if memory_summary.empty:
    display(Markdown('### CUDA memory'))
    display(Markdown('_Not available for this run. No `memory_measurements.csv` file or CUDA memory columns were found in the result rows._'))
else:
    display_table('CUDA memory summary', memory_summary, sort_by='delta_peak_cuda_mem_allocated_mb', ascending=True, save_name='memory_summary_display.csv')
    memory_plot = memory_summary.copy()
    memory_plot['run_label'] = memory_plot['model'] + ' | ' + memory_plot['prompt_mode'] + ' | ' + memory_plot['protocol']
    plot_barh(memory_plot, 'delta_peak_cuda_mem_allocated_mb', 'Mean delta peak CUDA allocated memory', 'memory_delta_peak_allocated_mb.png', lower_is_better=True)
    plot_barh(memory_plot, 'peak_cuda_mem_allocated_mb', 'Mean peak CUDA allocated memory', 'memory_peak_allocated_mb.png', lower_is_better=True)
    plot_barh(memory_plot, 'delta_peak_cuda_mem_allocated_mb_per_frame', 'Mean delta peak CUDA allocated memory per frame', 'memory_delta_peak_allocated_mb_per_frame.png', lower_is_better=True)


## 9. Per-Frame Behavior

In [ ]:
def plot_frame_curves(case_df, title, filename):
    if case_df.empty or 'frame_idx' not in case_df.columns or 'frame_dice' not in case_df.columns:
        return None
    fig, ax = plt.subplots(figsize=(12, 5.8))
    for label, group in case_df.sort_values('frame_idx').groupby('run_label', dropna=False):
        ax.plot(group['frame_idx'], group['frame_dice'], marker='o', markersize=2.7, linewidth=1.3, label=str(label))
    if 'anchor_frame' in case_df.columns:
        anchors = sorted({int(x) for x in case_df['anchor_frame'].dropna().tolist() if x >= 0})
        for anchor in anchors[:6]:
            ax.axvline(anchor, linestyle='--', linewidth=1.0, color='#111827', alpha=0.35)
    ax.set_title(title)
    ax.set_xlabel('frame index')
    ax.set_ylabel('frame Dice')
    ax.set_ylim(-0.03, 1.03)
    ax.legend(loc='center left', bbox_to_anchor=(1.01, 0.5), fontsize=8)
    return save_fig(fig, filename)


if per_frame.empty:
    display(Markdown('_Per-frame metrics are not available for this run._'))
else:
    worst_frames = per_frame.dropna(subset=['frame_dice']).sort_values('frame_dice', ascending=True).head(50)
    display_table('Lowest per-frame Dice rows', worst_frames, columns=['dataset_name', 'case_id', 'frame_idx', 'model', 'prompt_mode', 'protocol', 'frame_dice', 'frame_iou', 'anchor_frame'], sort_by='frame_dice', ascending=True, save_name='worst_per_frame_rows.csv')

    if PRIMARY_SCORE in ok_results.columns and not ok_results.empty:
        case_scores = ok_results.groupby(CASE_COLS, dropna=False)[PRIMARY_SCORE].mean().reset_index().sort_values(PRIMARY_SCORE, ascending=True)
        selected_cases = case_scores.head(int(MAX_PER_FRAME_CASES))
    else:
        selected_cases = per_frame.groupby(CASE_COLS, dropna=False).size().reset_index(name='rows').head(int(MAX_PER_FRAME_CASES))
    display_table('Selected cases for per-frame curves', selected_cases, save_name='selected_per_frame_cases.csv')

    for _, case_row in selected_cases.iterrows():
        dataset_name = str(case_row['dataset_name'])
        case_id = str(case_row['case_id'])
        case_df = per_frame[(per_frame['dataset_name'].astype(str) == dataset_name) & (per_frame['case_id'].astype(str) == case_id)].copy()
        if case_df.empty:
            continue
        title = f'Per-frame Dice: {dataset_name} / {case_id}'
        filename = f'per_frame_dice_{safe_name(dataset_name)}_{safe_name(case_id)}.png'
        plot_frame_curves(case_df, title, filename)


## 10. Failure and Audit Appendix

In [ ]:
display_table('Failed or non-ok result rows', failed_results, columns=['dataset_name', 'case_id', 'model', 'prompt_mode', 'protocol', 'status', 'error'], n=TOP_FAILURE_ROWS, save_name='failed_result_rows.csv')

if prompt_audit.empty:
    display(Markdown('_Prompt audit is not available for this run._'))
else:
    audit_work = prompt_audit.copy()
    audit_bad_mask = ~audit_work['status'].astype(str).str.lower().eq('ok') if 'status' in audit_work.columns else pd.Series(False, index=audit_work.index)
    if 'strict_equal_protocol' in audit_work.columns:
        strict_text = audit_work['strict_equal_protocol'].astype(str).str.lower()
        audit_bad_mask = audit_bad_mask | ~strict_text.isin(['true', '1', '1.0'])
    audit_bad = audit_work[audit_bad_mask].copy()
    display_table('Prompt audit rows needing attention', audit_bad, columns=['dataset_name', 'case_id', 'model', 'prompt_mode', 'protocol', 'status', 'error', 'number_of_prompted_frames', 'sequence_length', 'strict_equal_protocol'], n=TOP_FAILURE_ROWS, save_name='prompt_audit_attention_rows.csv')

if api_probe.empty:
    display(Markdown('_API probe is not available for this run._'))
else:
    display_table('API probe summary', api_probe, columns=['name', 'family', 'enabled', 'status', 'error', 'predictor_type', 'load_time_sec', 'checkpoint'], n=100, save_name='api_probe_summary.csv')

display(Markdown(f'Analysis complete. Tables and figures were written to `{ANALYSIS_OUTPUT_DIR}`.'))
